In [8]:
import sys
print("Путь к Python:", sys.executable)

Путь к Python: /Users/egormuhaev/Documents/Проекты/monitoring-construction-site/.venv/bin/python


In [ ]:
from pathlib import Path

import numpy as np
import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from torchvision.ops import box_iou, nms


def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return torch.serialization.load(*args, **kwargs)


torch.load = _torch_load_compat

import yolov5

weights = Path("weights/best.pt")
model_path = str(weights.resolve()) if weights.exists() else "uisikdag/yapimakine"
model = yolov5.load(model_path)
model.iou = 0.45
model.max_det = 1000

if torch.backends.mps.is_available():
    model.to("mps")
elif torch.cuda.is_available():
    model.to("cuda")

# Mobile_crane — это и автокран, и башенный (модель его так размечает).
CLASS_CONF = {
    "Worker": 0.25,
    "Safety helmet": 0.28,
    "Reflective vest": 0.25,
    "Dumb_truck": 0.40,
    "Loader": 0.50,
    "Excavator": 0.35,
    "Mobile_crane": 0.22,
    "Bull_dozer": 0.40,
    "Grader": 0.40,
    "Roller": 0.45,
}

CLASS_LABELS = {
    "Mobile_crane": "Crane",
}

CLASS_COLORS = {
    "Worker": "#22c55e",
    "Safety helmet": "#eab308",
    "Reflective vest": "#a3e635",
    "Dumb_truck": "#ef4444",
    "Loader": "#f97316",
    "Excavator": "#f59e0b",
    "Mobile_crane": "#38bdf8",
    "Bull_dozer": "#fb7185",
    "Grader": "#c084fc",
    "Roller": "#2dd4bf",
}

EQUIPMENT = {
    "Bull_dozer",
    "Dumb_truck",
    "Excavator",
    "Grader",
    "Loader",
    "Mobile_crane",
    "Roller",
}


def _empty_pred(device=None):
    return torch.zeros((0, 6), device=device)


def _predict_full(image, size, conf, augment=False):
    model.conf = conf
    model.iou = 0.45
    pred = model(image, size=size, augment=augment).pred[0]
    return pred if pred is not None else _empty_pred()


def _tile_starts(length, tile, overlap):
    tile = min(tile, length)
    stride = max(1, int(tile * (1 - overlap)))
    starts = list(range(0, max(length - tile, 0) + 1, stride))
    last = length - tile
    if not starts or starts[-1] != last:
        starts.append(max(0, last))
    return sorted(set(starts))


def _predict_tiles(image, tile=960, overlap=0.25, conf=0.12, infer_size=640):
    width, height = image.size
    model.conf = conf
    model.iou = 0.45
    parts = []
    for y in _tile_starts(height, tile, overlap):
        for x in _tile_starts(width, tile, overlap):
            tw, th = min(tile, width - x), min(tile, height - y)
            crop = image.crop((x, y, x + tw, y + th))
            pred = model(crop, size=infer_size).pred[0]
            if pred is None or len(pred) == 0:
                continue
            shifted = pred.clone()
            shifted[:, [0, 2]] += x
            shifted[:, [1, 3]] += y
            shifted[:, 0].clamp_(0, width)
            shifted[:, 2].clamp_(0, width)
            shifted[:, 1].clamp_(0, height)
            shifted[:, 3].clamp_(0, height)
            parts.append(shifted)
    if not parts:
        return _empty_pred()
    return torch.cat(parts, dim=0)


def _merge_nms(pred, iou=0.4):
    if pred is None or len(pred) == 0:
        return _empty_pred(getattr(pred, "device", None))
    keep = nms(pred[:, :4], pred[:, 4], iou)
    return pred[keep]


def _paint_stats(image, box):
    x1, y1, x2, y2 = [int(max(0, v)) for v in box]
    x2 = min(image.width, max(x1 + 1, x2))
    y2 = min(image.height, max(y1 + 1, y2))
    arr = np.asarray(image.crop((x1, y1, x2, y2)).convert("HSV"))
    hue = arr[:, :, 0].astype(int)
    sat = arr[:, :, 1].astype(int)
    val = arr[:, :, 2].astype(int)
    yellow = (hue >= 18) & (hue <= 50) & (sat >= 60) & (val >= 70)
    orange = (hue >= 8) & (hue <= 22) & (sat >= 70) & (val >= 70)
    red = ((hue <= 10) | (hue >= 245)) & (sat >= 80) & (val >= 60)
    green = (hue >= 50) & (hue <= 105) & (sat >= 70) & (val >= 50)
    paint = float((yellow | orange | red | green).mean())
    return paint, float(sat.mean()), float(val.mean())


def _keep_row(row, image):
    names = model.names
    x1, y1, x2, y2 = [float(v) for v in row[:4]]
    score = float(row[4])
    name = names[int(row[5])]
    img_w, img_h = image.size
    bw, bh = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    ar = bw / bh
    area = (bw * bh) / (img_w * img_h)
    rel_h = bh / img_h
    paint, _, val_mean = _paint_stats(image, (x1, y1, x2, y2))
    thresh = CLASS_CONF.get(name, 0.25)

    tower_like = (
        name == "Mobile_crane"
        and ar <= 0.7
        and area <= 0.08
        and y1 < 0.35 * img_h
        and score >= 0.14
    )
    if score < thresh and not tower_like:
        return False
    if min(bw, bh) < 14:
        return False
    if area > 0.16:
        return False
    if name == "Loader":
        if paint < 0.06:
            return False
        if ar >= 1.65 and rel_h <= 0.10 and paint < 0.18:
            return False
    if name == "Mobile_crane" and y1 > 0.42 * img_h and ar > 1.25:
        return False
    if name == "Worker" and ar > 0.85 and paint < 0.10 and area > 0.004:
        return False
    if name == "Worker" and min(bw, bh) >= 80 and y1 < 0.12 * img_h:
        return False
    if name == "Safety helmet" and paint < 0.03 and val_mean > 150:
        return False
    return True


def _filter_pred(pred, image):
    if pred is None or len(pred) == 0:
        return _empty_pred(getattr(pred, "device", None))
    kept = [row for row in pred if _keep_row(row, image)]
    if not kept:
        return _empty_pred(pred.device)
    pred = torch.stack(kept)
    names = model.names
    boxes = pred[:, :4]
    scores = pred[:, 4]
    classes = [names[int(row[5])] for row in pred]

    drop = torch.zeros(len(pred), dtype=torch.bool, device=pred.device)
    ious = box_iou(boxes, boxes)
    for i, cls in enumerate(classes):
        if cls != "Loader":
            continue
        for j, other in enumerate(classes):
            if i == j or other not in EQUIPMENT or other == "Loader":
                continue
            if float(ious[i, j]) >= 0.25 and float(scores[j]) >= float(scores[i]) - 0.05:
                drop[i] = True
                break

    for i, cls in enumerate(classes):
        if cls != "Mobile_crane" or drop[i]:
            continue
        hi = float(pred[i, 3] - pred[i, 1])
        for j, other in enumerate(classes):
            if i >= j or other != "Mobile_crane" or drop[j]:
                continue
            if float(ious[i, j]) < 0.2:
                continue
            hj = float(pred[j, 3] - pred[j, 1])
            if hi >= hj:
                drop[j] = True
            else:
                drop[i] = True
    return pred[~drop]


def infer(image, mode="hq", size=None, conf=None, tiles=None, augment=None):
    """mode='fast' — 640 / 0.25. mode='hq' — 640+TTA, 1280, тайлы и анти-FP фильтры."""
    width, height = image.size
    long_side = max(width, height)

    if mode == "fast":
        size = 640 if size is None else size
        conf = 0.25 if conf is None else conf
        use_tiles = False if tiles is None else tiles
        use_augment = False if augment is None else augment
        use_filters = False
        parts = [_predict_full(image, size=size, conf=conf, augment=use_augment)]
    else:
        hi = (1280 if long_side >= 900 else 960) if size is None else size
        conf = 0.12 if conf is None else conf
        use_tiles = (long_side >= 900) if tiles is None else tiles
        use_augment = True if augment is None else augment
        use_filters = True
        parts = [
            _predict_full(image, size=640, conf=0.15, augment=use_augment),
            _predict_full(image, size=hi, conf=conf, augment=False),
        ]
        if use_augment:
            parts.append(_predict_full(image, size=hi, conf=conf, augment=True))

    if use_tiles:
        tile = 960 if long_side < 2200 else 1280
        parts.append(_predict_tiles(image, tile=tile, overlap=0.25, conf=conf))

    parts = [p for p in parts if p is not None and len(p)]
    if not parts:
        return _empty_pred()
    pred = torch.cat(parts, dim=0)
    pred = _merge_nms(pred, iou=0.4)
    if use_filters:
        pred = _filter_pred(pred, image)
    return pred


def _label_name(name):
    return CLASS_LABELS.get(name, name)


def _draw(image, pred):
    vis = image.copy()
    draw = ImageDraw.Draw(vis)
    try:
        font = ImageFont.truetype("Arial.ttf", 16)
    except OSError:
        font = ImageFont.load_default()
    names = model.names
    for row in pred:
        x1, y1, x2, y2 = [float(v) for v in row[:4]]
        score = float(row[4])
        name = names[int(row[5])]
        color = CLASS_COLORS.get(name, "#22c55e")
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        label = f"{_label_name(name)} {score:.0%}"
        ty = max(0, y1 - 18)
        tb = draw.textbbox((x1, ty), label, font=font)
        draw.rectangle([tb[0] - 2, tb[1] - 1, tb[2] + 2, tb[3] + 1], fill=color)
        draw.text((x1, ty), label, fill="black", font=font)
    return vis


def _to_detections(pred):
    names = model.names
    detections = []
    print(f"✅ Всего обнаружено объектов: {len(pred)}")
    print("-" * 45)
    for i, row in enumerate(pred):
        x1, y1, x2, y2 = [float(v) for v in row[:4]]
        score = float(row[4])
        class_name = names[int(row[5])]
        shown = _label_name(class_name)
        print(f"Объект #{i + 1}: {shown.upper()}")
        print(f"  Уверенность: {score:.2%}")
        print(f"  Координаты: x1={x1:.1f}, y1={y1:.1f}, x2={x2:.1f}, y2={y2:.1f}")
        print("-" * 45)
        detections.append({"class": class_name, "label": shown, "score": score, "box": [x1, y1, x2, y2]})
    return detections


def detect(
    image_path,
    mode="hq",
    size=None,
    conf=None,
    tiles=None,
    augment=None,
    save=True,
    show=True,
    out_dir="results",
):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    pred = infer(image, mode=mode, size=size, conf=conf, tiles=tiles, augment=augment)
    detections = _to_detections(pred)
    vis = _draw(image, pred)

    if save:
        out_dir = Path(out_dir)
        out_dir.mkdir(exist_ok=True)
        suffix = "" if mode == "hq" else f"_{mode}"
        out_path = out_dir / f"result_{image_path.stem}{suffix}.jpg"
        vis.save(out_path)
        print(f"📁 Результат сохранён: {out_path}")

    if show:
        display(vis)
    return detections


def compare_detect(image_path, save=True, show=True):
    """Сравнение baseline (640) и HQ на одном кадре."""
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    fast_pred = infer(image, mode="fast")
    hq_pred = infer(image, mode="hq")

    print(f"{image_path}  {image.size[0]}×{image.size[1]}")
    print(f"fast: {len(fast_pred)} объектов | HQ: {len(hq_pred)} объектов")
    print("-" * 45)
    names = model.names

    def _brief(pred):
        from collections import Counter

        counts = Counter(_label_name(names[int(row[5])]) for row in pred)
        return ", ".join(f"{k}×{v}" for k, v in counts.most_common()) or "—"

    print(f"fast: {_brief(fast_pred)}")
    print(f"HQ:   {_brief(hq_pred)}")

    vis_fast = _draw(image, fast_pred)
    vis_hq = _draw(image, hq_pred)
    gap = 12
    canvas = Image.new("RGB", (vis_fast.width * 2 + gap, vis_fast.height + 36), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), "fast: size=640, conf=0.25", fill="black")
    draw.text((vis_fast.width + gap + 8, 8), "HQ: 640+TTA + 1280 + tiles + filters", fill="black")
    canvas.paste(vis_fast, (0, 36))
    canvas.paste(vis_hq, (vis_fast.width + gap, 36))

    if save:
        out_dir = Path("results")
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / f"compare_{image_path.stem}.jpg"
        canvas.save(out_path)
        print(f"📁 Сравнение сохранено: {out_path}")

    if show:
        display(canvas)

    def _as_list(pred):
        return [
            {
                "class": names[int(row[5])],
                "label": _label_name(names[int(row[5])]),
                "score": float(row[4]),
                "box": [float(v) for v in row[:4]],
            }
            for row in pred
        ]

    return {"fast": _as_list(fast_pred), "hq": _as_list(hq_pred)}


In [ ]:
detect("dataset/photo3.webp")

In [ ]:
detect("dataset/photo4.jpg")

In [ ]:
detect('dataset/photo5.jpg')

In [ ]:
detect('dataset/photo6.jpg')

Сравнение baseline и HQ на проблемных кадрах (`photo5`, `photo6`):


In [ ]:
compare_detect("dataset/photo5.jpg")


In [ ]:
compare_detect("dataset/photo6.jpg")
